# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library, following best practices for reproducibility and referencing by Croissant `@id` fields.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

This dataset contains tabular records on clinicopathological and molecular characteristics of second primary colorectal cancer in cancer survivors.

In [ ]:
# If mlcroissant is not installed, install it:
!pip install mlcroissant

## 1. Data Loading
We use `mlcroissant` to load both metadata and records from the dataset defined by the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset (includes schema and references to data)
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review the available record sets, fields, and their corresponding `@id` fields as defined in the Croissant schema.

We will enumerate all record sets and list their available fields (with IDs), which is necessary to reference elements in later steps.

In [ ]:
# Inspect record sets available in the dataset
print("Available record sets and their fields (@id):\n")
record_sets = []
for record_set in dataset.record_sets:
    print(f"- Record Set: {record_set['@id']} (name: {record_set.get('name', 'n/a')})")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    if fields:
        print("  Fields:")
        for field in fields:
            field_id = field.get('@id', '(none)')
            field_name = field.get('name', 'n/a')
            print(f"    - {field_id} (name: {field_name})")
    else:
        print("  No fields found.")
    record_sets.append(record_set['@id'])

## 3. Data Extraction
Load the data from the main record set(s) into pandas DataFrames. All Croissant data elements are referenced by their `@id`.

First, we list the available record set IDs (from above), then load each to a DataFrame for further analysis.

In [ ]:
# 1. Gather record set IDs
record_set_ids = record_sets  # from previous cell
print(f"Record sets found: {record_set_ids}\n")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading data from record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
    except Exception as e:
        print(f"  Could not load data for {record_set_id}: {e}")

# Display columns in the first record set with data
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nColumns in {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print('No dataframes loaded. Check dataset schema for available record sets.')

## 4. Exploratory Data Analysis (EDA)
We will process and analyze the main table by referencing fields by their `@id`, performing filtering, normalization, and grouping operations where possible.

*Note:* All variable references are by `@id`, and if you want to reuse this notebook for alternative fields, substitute the relevant `@id` values.

In [ ]:
# For demonstration, select one record set with data
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Selected record set: {record_set_id}")
    # List available columns and types
    print("Columns and sample data:")
    print(df.dtypes)
    print(df.head(2))
    # Try to find a numeric field for demonstration
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if not numeric_field_id:
        # Fallback: try converting columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if pd.api.types.is_numeric_dtype(df[col]) and df[col].notnull().any():
                    numeric_field_id = col
                    break
            except Exception:
                continue
    if numeric_field_id:
        print(f"\nChosen numeric field (by @id): {numeric_field_id}\n")
        # Set a threshold (example: 10)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize field
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try grouping by a field, e.g., a categorical field
        # For demonstration, pick the first non-numeric column, or 'Sex' if found
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                if col.lower() == 'sex':
                    break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print('No categorical group field found for grouping.')
    else:
        print("No numeric field found for EDA in this record set.")
else:
    print('No dataframes available for EDA.')

## 5. Visualization
Create visualizations of the numeric fields or categorical distributions, referencing fields by their `@id`.

Here, we plot a histogram for the selected numeric field and a bar plot for a key categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Numeric field (as identified above)
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if not numeric_field_id:
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if pd.api.types.is_numeric_dtype(df[col]) and df[col].notnull().any():
                    numeric_field_id = col
                    break
            except Exception:
                continue
    if numeric_field_id:
        plt.figure(figsize=(8,5))
        sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
        plt.title(f"Histogram of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

    # Bar plot for categorical field (e.g., 'Sex' or first object dtype)
    categorical_field_id = None
    for col in df.columns:
        if df[col].dtype == object and col.lower() == 'sex':
            categorical_field_id = col
            break
    if not categorical_field_id:
        for col in df.columns:
            if df[col].dtype == object:
                categorical_field_id = col
                break
    if categorical_field_id:
        plt.figure(figsize=(7,5))
        sns.countplot(data=df, x=categorical_field_id)
        plt.title(f"Distribution of {categorical_field_id}")
        plt.xlabel(categorical_field_id)
        plt.ylabel("Count")
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion

In this notebook, we:
- Loaded and inspected a Croissant-formatted dataset describing second primary colorectal cancer in survivors, using only entity `@id` fields for references.
- Extracted tabular data and performed initial data processing and cleaning steps (filtering, normalizing, and grouping), always referencing data elements by `@id`.
- Created visualizations for numeric and categorical fields, referencing by `@id`.

This workflow demonstrates how `mlcroissant` enables transparent, reproducible, and FAIR handling of complex datasets with rich schema and annotations. Further domain-specific analyses can build on this structure.